In [1]:
from _utils import *

# Configure logging
log_dir = "../logs"
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

log_file = os.path.join(log_dir, f"model_db_download_errors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Load dataset from Excel
df = pd.read_excel("../annotations/model_db_annotations.xlsx")

# Define output directories and files
data_dir = "../data"
output_json = os.path.join(data_dir, "model_db_metadata.json")


/home/imc33/.conda/envs/mod-annotation/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


FileNotFoundError: [Errno 2] No such file or directory: 'gsheet-creds.json'

In [ ]:

# Create data dictionary to document the data structure
data_dictionary = {
    "dataset_description": "Metadata collection of neuron models from ModelDB including download URLs and file information",
    "created_date": datetime.now().strftime("%Y-%m-%d"),
    "source": "ModelDB - A database of computational neuroscience models",
    "fields": {
        "row_id": "Original row identifier from the annotations spreadsheet",
        "file_hash": "Hash value of the file content for identification",
        "raw_sha": "SHA hash of the raw file content",
        "count": "Occurrence count in the dataset",
        "url": "Original ModelDB URL for the file",
        "download_url": "Direct download URL constructed from the original URL",
        "filename": "Name of the file extracted from the URL",
        "content": "Raw content of the downloaded file when available",
        "error_code": "Error code if download failed, null for successful downloads",
        "download_date": "Timestamp when the metadata was recorded"
    },
    "entries": []  # Will store the actual data
}

# Failed downloads counter
failed_download_count = 0

# Create data directory if it doesn't exist
if not os.path.exists(data_dir):
    print(f"Creating directory: {data_dir}")
    os.makedirs(data_dir)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "filename": file_path,  # Store the filename separately
        "content": None,  # Default to None (indicating missing content)
        "error_code": None,  # New field for failed downloads
        "download_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # Add timestamp
    }
    
    if not direct_url:
        error_msg = f"Invalid URL format: {url}"
        print(f"Skipping invalid URL: {url}")
        logging.error(f"Row ID {row_id}: {error_msg}")
        file_entry["error_code"] = "Invalid URL"
        failed_download_count += 1
        data_dictionary["entries"].append(file_entry)  # Store even failed ones
        return
        
    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        error_msg = f"HTTP Error {response.status_code}: {http_err}"
        print(f"Failed to fetch {direct_url} - {error_msg}")
        logging.error(f"Row ID {row_id}, URL: {direct_url} - {error_msg}")
        file_entry["error_code"] = str(response.status_code)
        failed_download_count += 1
    except requests.exceptions.RequestException as e:
        error_msg = f"Request Error: {str(e)}"
        print(f"Failed to fetch {direct_url}: {e}")
        logging.error(f"Row ID {row_id}, URL: {direct_url} - {error_msg}")
        file_entry["error_code"] = "Request Error"
        failed_download_count += 1
        
    data_dictionary["entries"].append(file_entry)  # Store the file entry (success or failure)

# Select rows from the dataset
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Log the start of processing
logging.info(f"Starting metadata collection for {len(df_subset)} ModelDB entries")
print(f"Starting metadata collection for {len(df_subset)} ModelDB entries")

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Collecting ModelDB metadata"):
    fetch_mod_metadata(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Add summary statistics to the data dictionary
data_dictionary["total_entries"] = len(data_dictionary["entries"])
data_dictionary["successful_downloads"] = len(data_dictionary["entries"]) - failed_download_count
data_dictionary["failed_downloads"] = failed_download_count

# Save the complete data dictionary to JSON
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(data_dictionary, json_file, indent=4)

# Log completion
completion_message = f"\nModelDB metadata saved in {output_json}"
print(completion_message)
logging.info(completion_message)

stats_message = f"Total entries: {data_dictionary['total_entries']}, " \
               f"Successful downloads: {data_dictionary['successful_downloads']}, " \
               f"Failed downloads: {failed_download_count}"
print(stats_message)
logging.info(stats_message)

if failed_download_count > 0:
    print(f"Some metadata collections failed. Check log file: {log_file}")
    logging.info("Download process completed with errors")
else:
    logging.info("Download process completed successfully")